# Rate-of-Rise als Umkehrproblem
### Amortisierte Bayes-Inferenz aus einer einzelnen Druckanstiegskurve – mit selbst­geplanter Folgemessung

Eine Rate-of-Rise-Messung liefert eine Kurve $P(t)$ bei geschlossenem Ventil. Die Frage dahinter ist ein **Inversionsproblem**: Welcher Mechanismus hat sie erzeugt – ein echtes Leck (linear, unbegrenzt) oder ein gebundenes Reservoir / virtuelles Leck (sättigend)? Der klassische Weg fixiert *ein* Modell und fittet es; die Modellwahl trifft man vorab, und konkurrierende Hypothesen bleiben außen vor.

Hier wird stattdessen der Posterior $p(\theta \mid P)$ über die physikalischen Parameter direkt geschätzt – **simulationsbasiert und amortisiert**: ein Netz wird einmal an simulierten Kurven trainiert und liefert danach für jede neue Messung in Millisekunden die volle Verteilung, ohne Likelihood-Auswertung und ohne Modellannahme im Fit. Zwei Dinge, die mit einem klassischen $\chi^2$-Fit strukturell nicht gehen, stehen im Zentrum:

1. **Der Posterior meldet seine eigene Entartung.** Wo Leck und Reservoir aus einer Kurve grundsätzlich nicht trennbar sind, ergibt sich kein Fehlwert, sondern eine sichtbar entartete Verteilung.
2. **Der Algorithmus plant die nächste Messung.** Über den erwarteten Informationsgewinn wird bestimmt, welche Folgemessung die Entartung am stärksten bricht – hier: bei welcher Temperatur.

Alles läuft auf CPU; das Training dauert wenige Minuten. Benötigt `torch` und `scipy` (in Colab vorhanden).

In [ ]:
import numpy as np, torch, torch.nn as nn, time
import matplotlib.pyplot as plt
from scipy.stats import kstest
torch.manual_seed(0); np.random.seed(0)

# ---------------- Vorwärtsmodell ----------------
# Ventil zu:  V dP/dt = Q_leck + Q_res(t,T)
#   echtes Leck:            linear,  Steigung  s = Q_leck/V          (T-unabhängig)
#   Reservoir/virt. Leck:   P_res = h*(1 - e^{-t/tau(T)}),  h = N/V   (sättigt)
#   Freisetzungsrate 1/tau(T) folgt Arrhenius: schnell heiß, träge kalt
kB = 8.617e-5                 # eV/K
Ea, k0 = 0.49, 268302.0       # so kalibriert, dass tau(293K)=1000s, tau(373K)=16s
def tau_of_T(T): return np.exp(Ea/(kB*np.asarray(T,float)))/k0

T_WIN, NT = 100.0, 50
tt = np.linspace(1.0, T_WIN, NT).astype(np.float32)
SLOPE_MAX, H_MAX, NOISE = 0.35, 30.0, 0.3   # Priors: s~U(0,.35), h~U(0,30); Messrauschen

def sim_batch(S, H, T, rng):
    S = np.asarray(S)[:,None]; H = np.asarray(H)[:,None]; tau = tau_of_T(T)[:,None]
    return (S*tt[None] + H*(1-np.exp(-tt[None]/tau)) + rng.normal(0,NOISE,(len(S),NT))).astype(np.float32)

print("tau(293K) = %.0f s  (>> Fenster -> nahezu linear, entartet)" % tau_of_T([293])[0])
print("tau(373K) = %.0f s  (<< Fenster -> gesättigt, auflösend)"    % tau_of_T([373])[0])

## Die Mehrdeutigkeit, um die es geht

Bei **kalter** Messung ($\tau \gg$ Fenster) trägt das Reservoir nur seinen anfänglichen, quasi-linearen Ast bei – ein echtes Leck und ein großes, träges Reservoir erzeugen dann fast dieselbe Kurve. Bei **warmer** Messung ($\tau \ll$ Fenster) sättigt das Reservoir innerhalb des Fensters und trennt sich in der Form klar vom Leck. Genau dieser Temperaturhebel wird später zum Werkzeug.

In [ ]:
rng = np.random.default_rng(1)
faelle = [(0.30, 0.0, 'reines Leck'), (0.05, 25.0, 'reines Reservoir'), (0.17, 12.0, 'Mischung')]
fig, ax = plt.subplots(1, 2, figsize=(11,4), sharey=True)
for T, a, titel in [(293, ax[0], 'kalt (293 K)'), (373, ax[1], 'warm (373 K)')]:
    for s, h, lab in faelle:
        P = sim_batch([s],[h],[T], np.random.default_rng(2))[0]
        a.plot(tt, P, label=lab)
    a.set_title(titel); a.set_xlabel('t [s]'); a.grid(alpha=.3); a.legend()
ax[0].set_ylabel('P [a.u.]')
plt.suptitle('Dieselben drei Systeme: kalt kaum unterscheidbar, warm klar getrennt')
plt.tight_layout(); plt.show()

## Amortisierter neuronaler Posterior

Ein Mixture-Density-Netz bildet $x = (P(t),\, T) \mapsto p(\theta\mid x)$ als Gauß-Mischung ab (hier $K=8$ Komponenten – die Mischung erlaubt mehrgipflige, gekrümmte Posteriors, wie sie bei Entartung auftreten). Trainiert wird auf Paaren $(\theta,\,x)$ aus dem Vorwärtsmodell; die Temperatur ist Teil der Eingabe, sodass **ein** Netz jede Sondiertemperatur bedient. Nach dem Training ist Inferenz reine Vorwärtsauswertung.

In [ ]:
class MDN(nn.Module):
    def __init__(self, d_in, d_out=2, K=8, hid=128):
        super().__init__(); self.K, self.d = K, d_out
        self.body = nn.Sequential(nn.Linear(d_in,hid), nn.ReLU(), nn.Linear(hid,hid), nn.ReLU())
        self.pi = nn.Linear(hid,K); self.mu = nn.Linear(hid,K*d_out); self.logs = nn.Linear(hid,K*d_out)
    def forward(self, x):
        h = self.body(x)
        return (torch.log_softmax(self.pi(h),1),
                self.mu(h).view(-1,self.K,self.d),
                self.logs(h).view(-1,self.K,self.d).clamp(-6,3))
    def nll(self, x, y):
        lp, mu, ls = self(x); y = y.unsqueeze(1)
        comp = (-0.5*(((y-mu)/ls.exp())**2) - ls - 0.5*np.log(2*np.pi)).sum(-1)
        return -(torch.logsumexp(lp+comp,1)).mean()
    def sample(self, x, n=3000):
        with torch.no_grad():
            lp, mu, ls = self(x); k = torch.multinomial(lp.exp().expand(n,-1),1).squeeze(1)
            return (mu[0,k] + torch.randn_like(mu[0,k])*ls[0,k].exp()).numpy()

nth = lambda s,h: np.stack([s/SLOPE_MAX, h/H_MAX],-1).astype(np.float32)   # theta -> [0,1]^2
dn  = lambda z: z*np.array([SLOPE_MAX, H_MAX])                             # zurück

def make1(n, seed):
    rng = np.random.default_rng(seed)
    S = rng.uniform(0,SLOPE_MAX,n); H = rng.uniform(0,H_MAX,n); Ts = rng.uniform(293,373,n)
    X = np.concatenate([sim_batch(S,H,Ts,rng), ((Ts-293)/80.0)[:,None].astype(np.float32)], 1)
    return X, nth(S,H)

t0 = time.time()
X1, Y1 = make1(40000, 1)
mx, sx = X1.mean(0), X1.std(0)+1e-6
Xt, Yt = torch.tensor((X1-mx)/sx), torch.tensor(Y1)
net1 = MDN(NT+1, K=8)
opt = torch.optim.Adam(net1.parameters(), 1.5e-3)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 300)
for ep in range(300):
    perm = torch.randperm(len(Xt))
    for b in range(0, len(Xt), 1024):
        i = perm[b:b+1024]; opt.zero_grad(); loss = net1.nll(Xt[i], Yt[i]); loss.backward(); opt.step()
    sch.step()
print(f"NPE_1 trainiert in {time.time()-t0:.0f}s  (NLL {float(loss.detach()):.3f})")

def posterior1(P, T, n=3000):
    x = np.concatenate([P, [(T-293)/80.0]]).astype(np.float32)
    return dn(net1.sample(torch.tensor(((x-mx)/sx)[None]), n))

## Kalibrierung: ist dem Posterior zu trauen?

Ein Posterior ist nur brauchbar, wenn seine Unsicherheiten *kalibriert* sind. Der Test dafür ist **simulationsbasierte Kalibrierung (SBC)**: Zieht man Parameter aus dem Prior, simuliert eine Messung und bestimmt den Rang des wahren Werts in den Posterior-Stichproben, müssen diese Ränge bei korrekter Inferenz gleichverteilt sein. Abweichungen von der Gleichverteilung zeigen Über- oder Unterkonfidenz.

In [ ]:
rngS = np.random.default_rng(21); ranks = []
for _ in range(300):
    s = rngS.uniform(0,SLOPE_MAX); h = rngS.uniform(0,H_MAX); Tq = rngS.uniform(293,373)
    P = sim_batch([s],[h],[Tq], rngS)[0]; ps = posterior1(P, Tq, 400)
    ranks.append([(ps[:,0]<s).mean(), (ps[:,1]<h).mean()])
ranks = np.array(ranks)

fig, ax = plt.subplots(1, 2, figsize=(10,3.4))
for j,(nm) in enumerate(['Leckrate s', 'Reservoir h']):
    ax[j].hist(ranks[:,j], bins=12, color='steelblue', alpha=.8, edgecolor='w')
    ax[j].axhline(len(ranks)/12, color='crimson', ls='--')
    p = kstest(ranks[:,j],'uniform').pvalue
    ax[j].set_title(f'{nm}: SBC-Rang  (KS p={p:.2f})'); ax[j].set_xlabel('Rang')
plt.suptitle('Ränge ~ gleichverteilt  ⇒  Unsicherheiten kalibriert'); plt.tight_layout(); plt.show()

## Die Entartung wird sichtbar

Drei Messungen bei **293 K**: ein klares Leck, ein klares Reservoir und ein mehrdeutiger Mischfall. Der Posterior im $(s,h)$-Raum spricht für sich – im Mischfall spannt er die charakteristische antikorrelierte „Banane“ auf: viel Leck + wenig Reservoir oder wenig Leck + viel Reservoir passen gleich gut. Das ist keine Schwäche des Schätzers, sondern die ehrliche Aussage, dass die kalte Kurve die Frage nicht beantwortet.

In [ ]:
szenarien = [(0.30, 2.0, 'klares Leck'), (0.04, 24.0, 'klares Reservoir'), (0.10, 20.0, 'mehrdeutig')]
fig, ax = plt.subplots(1, 3, figsize=(13,4))
for a,(s,h,titel) in zip(ax, szenarien):
    P = sim_batch([s],[h],[293], np.random.default_rng(7))[0]
    ps = posterior1(P, 293)
    a.scatter(ps[:,0], ps[:,1], s=4, alpha=.15, color='slateblue')
    a.scatter([s],[h], color='crimson', marker='x', s=120, label='wahr')
    a.set_xlim(0,SLOPE_MAX); a.set_ylim(0,H_MAX); a.set_xlabel('Leckrate s'); a.grid(alpha=.3)
    a.set_title(f'{titel}\n(r={np.corrcoef(ps.T)[0,1]:+.2f}, σ_h={ps[:,1].std():.1f})'); a.legend(loc='upper right')
ax[0].set_ylabel('Reservoir h')
plt.suptitle('Posterior bei 293 K: der Mischfall zeigt die Leck–Reservoir-Entartung offen an')
plt.tight_layout(); plt.show()

## Die Folgemessung plant sich selbst

Steht die kalte Messung, ist die Frage: **welche Zusatzmessung bringt am meisten?** Formal maximiert man den erwarteten Informationsgewinn (EIG)
$$\mathrm{EIG}(T) = H\!\left[p(\theta\mid P_{\text{kalt}})\right] - \mathbb{E}_{P_T}\!\left[H\!\left[p(\theta\mid P_{\text{kalt}}, P_T)\right]\right].$$
Dazu wird ein zweiter, amortisierter Schätzer trainiert, der **beide** Kurven samt Sondiertemperatur verarbeitet. Der EIG wird über simulierte Sondierausgänge geschätzt, gezogen aus dem aktuellen Kalt-Posterior – ganz ohne die wahren Parameter zu kennen.

In [ ]:
def make2(n, seed):
    rng = np.random.default_rng(seed)
    S = rng.uniform(0,SLOPE_MAX,n); H = rng.uniform(0,H_MAX,n); Tp = rng.uniform(293,373,n)
    Xc = sim_batch(S,H,[293]*n if False else np.full(n,293.0), rng)
    Xp = sim_batch(S,H,Tp,rng)
    X  = np.concatenate([Xc, Xp, ((Tp-293)/80.0)[:,None].astype(np.float32)], 1)
    return X, nth(S,H)

t0 = time.time()
X2, Y2 = make2(40000, 3)
mx2, sx2 = X2.mean(0), X2.std(0)+1e-6
Xt2, Yt2 = torch.tensor((X2-mx2)/sx2), torch.tensor(Y2)
net2 = MDN(2*NT+1, K=8)
opt2 = torch.optim.Adam(net2.parameters(), 1.5e-3)
sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, 250)
for ep in range(250):
    perm = torch.randperm(len(Xt2))
    for b in range(0, len(Xt2), 1024):
        i = perm[b:b+1024]; opt2.zero_grad(); l = net2.nll(Xt2[i], Yt2[i]); l.backward(); opt2.step()
    sch2.step()
print(f"NPE_2 trainiert in {time.time()-t0:.0f}s  (NLL {float(l.detach()):.3f})")

def posterior2(Pc, Pp, Tp, n=3000):
    x = np.concatenate([Pc, Pp, [(Tp-293)/80.0]]).astype(np.float32)
    return dn(net2.sample(torch.tensor(((x-mx2)/sx2)[None]), n))

def entropie(post):                      # diff. Entropie einer Gauß-Näherung
    c = np.cov(post.T) + 1e-9*np.eye(2)
    return 0.5*np.log(np.linalg.det(2*np.pi*np.e*c))

In [ ]:
# aktueller Zustand: eine kalte, mehrdeutige Messung
s_true, h_true = 0.10, 20.0
rng = np.random.default_rng(7)
P_kalt = sim_batch([s_true],[h_true],[293], rng)[0]
post_kalt = posterior1(P_kalt, 293)
H_kalt = entropie(post_kalt)

# EIG über Sondiertemperatur
Tgrid = np.array([293, 313, 333, 353, 373.])
rngE = np.random.default_rng(11); eig = []
for Tp in Tgrid:
    hs = []
    for s,h in post_kalt[rngE.integers(0,len(post_kalt),40)]:      # Ausgänge ~ Kalt-Posterior
        Pp = sim_batch([s],[h],[Tp], rngE)[0]
        hs.append(entropie(posterior2(P_kalt, Pp, Tp, 1500)))
    eig.append((H_kalt - np.mean(hs))/np.log(2))                   # in Bit
eig = np.array(eig)

# empfohlene Messung tatsächlich durchführen
P_warm = sim_batch([s_true],[h_true],[373], rngE)[0]
post_komb = posterior2(P_kalt, P_warm, 373)

fig, ax = plt.subplots(1, 2, figsize=(12,4.2))
ax[0].plot(Tgrid, eig, 'o-', color='seagreen'); ax[0].set_xlabel('Sondiertemperatur [K]')
ax[0].set_ylabel('erwarteter Informationsgewinn [bit]'); ax[0].grid(alpha=.3)
ax[0].set_title('Der Algorithmus bewertet Folgemessungen und empfiehlt Aufheizen')
ax[1].scatter(post_kalt[:,0], post_kalt[:,1], s=5, alpha=.12, color='indianred', label='nur kalt')
ax[1].scatter(post_komb[:,0], post_komb[:,1], s=5, alpha=.3, color='seagreen', label='kalt + warm (373 K)')
ax[1].scatter([s_true],[h_true], color='black', marker='x', s=130, label='wahr')
ax[1].set_xlim(0,SLOPE_MAX); ax[1].set_ylim(0,H_MAX)
ax[1].set_xlabel('Leckrate s'); ax[1].set_ylabel('Reservoir h'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title('Die empfohlene Messung kollabiert den Posterior')
plt.tight_layout(); plt.show()

print(f"EIG: {dict(zip(Tgrid.astype(int), np.round(eig,1)))} bit")
print(f"nur kalt:      h = {post_kalt[:,1].mean():.1f} ± {post_kalt[:,1].std():.1f}")
print(f"kalt + warm:   h = {post_komb[:,1].mean():.1f} ± {post_komb[:,1].std():.1f}   (wahr {h_true})")
print(f"Informationsgewinn der empfohlenen Messung: {(H_kalt-entropie(post_komb))/np.log(2):.1f} bit")

---
### Einordnung und Grenzen

Das Vorwärtsmodell ist bewusst kompakt: ein konstantes Leck, ein einzelnes Reservoir mit Arrhenius-Freisetzung, additives Gauß-Rauschen, konstante Temperatur je Messung. Ein reales System bringt mehrere Desorptionszeitkonstanten, temperaturabhängige Leitwerte, Pumpdynamik und korreliertes Sensorrauschen mit – all das gehört ins Vorwärtsmodell, nicht in den Schätzer, und genau das ist der Punkt: Die Methode erbt jede Physik, die man zu simulieren bereit ist, und der Rest (Posterior, Kalibrierung, Versuchsplanung) bleibt unverändert.

Was hier konkret anders ist als beim klassischen Fit: Es wird **kein** Modell vorab ausgewählt, konkurrierende Mechanismen bleiben gemeinsam im Posterior, dessen **Entartung explizit sichtbar** wird, und die **informativste Folgemessung** ergibt sich aus dem Modell selbst statt aus Erfahrung. Der Preis ist ein Simulationsbudget vorab; der Gewinn ist amortisierte Inferenz in Echtzeit und eine quantitative Antwort auf die Frage „was messe ich als Nächstes?“.

**Methodische Referenzen:** Cranmer, Brehmer, Louppe, *The frontier of simulation-based inference* (PNAS 2020); Papamakarios & Murray, *Fast ε-free inference* (NeurIPS 2016) zur neuronalen Posterior-Schätzung; Talts et al., *Validating Bayesian inference algorithms with SBC* (2018); Lindley (1956) / Ryan et al. (2016) zur bayesschen Versuchsplanung.